In [ ]:
import pandas as pd

from ada_fsaudit_bridge import (
    att_sample,
    configure_environment,
    cvs_sample,
    load_dataset,
    lower_bound,
    mus_sample,
    set_notebook_context,
    upper_bound,
)
from scipy.stats import hypergeom

ADA_WORKSHOP_ID = "auxiliary-variables-and-stratification"
ADA_CHAPTER = "3"
ADA_NOTEBOOK_SOURCE = "notebooks/support/auxiliary-variables-and-stratification/support.Rmd"

def ada_set_context(exercise_ref: str) -> None:
    set_notebook_context(
        chapter=ADA_CHAPTER,
        exercise=exercise_ref,
        notebook=f'{ADA_WORKSHOP_ID}:{ADA_NOTEBOOK_SOURCE}',
    )

configure_environment()


## Exercise 3.1. The `inventoryData` file


In this Chapter we will work with the `inventoryData` data file, that serves as the sample frame for the *Case: Valuation of inventories*. The data file is included with the `FSaudit` package. The data file indicates for each of the 3,500 elements in the inventories population its `id`, a sequential number from 1 to 3,500, the book value $x$ is `bv`, and we have added an audit value $y =$ `av` to enable running simulations on the file to compare different sampling strategies.
Details of the data file can be obtained using base `R` functionality.


In [ ]:
from IPython.display import display
ada_set_context("3.1")
inventoryData = load_dataset("inventoryData")
inventoryData.columns.tolist()
len(inventoryData)
display(inventoryData["bv"].sum())
display(inventoryData.iloc[:6, :3])


## Exercise 3.2. Working with the CVS object


We first create a Classical Variables Sampling ("CVS") object. An object is a container that gathers all the data relevant to a particular sampling application. The `FSaudit` package recognizes three such objects. We work with the CVS object `cvs_obj` in this chapter, and with the attribute `att_obj` and MUS `mus_obj` objects in Chapter 5.
We create the CVS object `mySample` by running the `cvs_obj` function, and at the same time fill it with information on the required sample size `n`, the relevant book values `bv`, identifiers `id`, and the random seed number `seed`.


In [ ]:
ada_set_context("3.2")
mySample = cvs_sample(
    n=400,
    bv=inventoryData["bv"],
    id=inventoryData["item"],
    seed=12345,
)


The CVS object has 19 different attributes attached to it, that will be filled as we proceed from sample planning, through stratification and selection to evaluation. The following is a list of all attributes.


In [ ]:
ada_set_context("3.2")
mySample.field_names()


Some of these attributes are empty for now, for example `sample`.


In [ ]:
ada_set_context("3.2")
mySample.sample


In order to replicate the experiment in the book, you need to use the same random number seed as we did. Note that as of version 3.6, fundamental changes have been made in the random number generator. The default setting of the random number generator should normally not be changed, but for the benefit of readers who use an older version of *R* we set it to `Rounding`, which was the default prior to version 3.6.


In [ ]:
ada_set_context("3.2")
configure_environment(seed=12345)


We sample 400 units from the `inventoryData` dataset with the `select()` function. This effectively fills the `sample` attribute in `mySample`.


In [ ]:
ada_set_context("3.2")
mySample.select()
mySample.sample.head()


We audit the sampling units selected, and submit the audit values to the CVS object for the sample to be evaluated.


In [ ]:
ada_set_context("3.2")
audit_values = inventoryData.set_index("item").loc[mySample.sample["item"], "av"]
mySample.evaluate(av=audit_values)


The evaluation results are stored in the `evalResults` attribute of the CVS object. This attribute itself is a list of 17 different attributes.


In [ ]:
ada_set_context("3.2")
list(mySample.eval_results.keys())


## Exercise 3.3. Mean-per-unit estimator


In Section 4.1 we found an estimate of the population value $\hat{Y}_{MPU}$ = 7,561,859, and achieved precision of 684,415. These values are stored in the attribute `evalResults` of `mySample`.


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Most likely total audited amount mpu"]


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Achieved precision mpu"]


The prediction interval is stored in:


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Estimates"].iloc[3:5, 0]


In this call, the numbers in brackets refer to the 4th and 5th row of the Estimates table. The first column stores the results of the mean-per-unit estimator.
Alternatively, we can refer to the label of the relevant column.


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("mpu")]


## Exercise 3.4. The Regression Estimator


The results for the regression estimator are stored in a similar way.


In [ ]:
ada_set_context("3.4")
mySample.eval_results["Most likely total audited amount regression"]


In [ ]:
ada_set_context("3.4")
mySample.eval_results["Achieved precision regression"]


The prediction interval is determined by:


In [ ]:
ada_set_context("3.4")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("regr")]


## Exercise 3.5. The Difference Estimator


The results for the difference estimator are as follows.


In [ ]:
ada_set_context("3.5")
mySample.eval_results["Most likely total audited amount difference"]


In [ ]:
ada_set_context("3.5")
mySample.eval_results["Achieved precision difference"]


The prediction interval is determined by:


In [ ]:
ada_set_context("3.5")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("diff")]


## Exercise 3.6. The Ratio Estimator


Finally, the results of the ratio estimator.


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Most likely total audited amount ratio"]


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Achieved precision ratio"]


The prediction interval is determined by:


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("ratio")]


An overview of all estimates is given by


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Estimates"]


## Exercise 3.7. Using CVS with sporadic errrors


We start with setting up a cvs_obj object, and fill it with the parameters. We use the prefix `ar` for accounts receivable.


In [ ]:
ada_set_context("3.7")
accounts_receivable = load_dataset("accounts_receivable")
arSample = cvs_sample(
    bv=accounts_receivable["amount"],
    id=accounts_receivable["invoice"],
    n=100,
)


We select the sample, using a seed equal to 1.


In [ ]:
ada_set_context("3.7")
arSample.select(seed=1)


We look up the audit values of the selected sampling units.


In [ ]:
ada_set_context("3.7")
audit_values = accounts_receivable.set_index("invoice").loc[arSample.sample["item"], "av2"]


We then evaluate the sample and look at the `Estimates` attribute.


In [ ]:
ada_set_context("3.7")
arSample.evaluate(av=audit_values)
arSample.eval_results["Estimates"]


The number of differences is stored in the `#_Errors` attribute, under each of the estimation methods. This way, you can tell if precision was calculated with the sporadic error (k = 4 to 19) method, or using the standard CVS method.


In [ ]:
ada_set_context("3.7")
arSample.eval_results["Regression estimation"]["#_Errors"]


Most of the calculations presented in Section 4.7 are easy to follow, so we will not repeat them all in this workshop. Instead, we zoom in on some specific parts of the overall calculations.
The value of $M_U$ that is used in, for example, Equation 4.23 is obtained as follows.


In [ ]:
ada_set_context("3.7")
N = len(accounts_receivable)
m = 6
(m_u := upper_bound(k=m, popn=N, n=100, alpha=0.05) / N)


The effective degrees of freedom is $m - 1$. This explains the $t$ value used when calculating the confidence intervals.


In [ ]:
from scipy.stats import t
ada_set_context("3.7")
t.ppf(0.975, df=m - 1)


## Exercise 3.8. Stratification with equal recorded boundaries


We start with creating a CVS object.


In [ ]:
ada_set_context("3.8")
equal = cvs_sample(bv=inventoryData["bv"], id=inventoryData["item"])


We then stratify with the equal method (`stratMeth = equal`), and view the summary.


In [ ]:
ada_set_context("3.8")
equal.stratify(strata=3, stratMeth="equal")
equal.field("stratSumm")


## Exercise 3.9. Stratification with the cumulative method


In [ ]:
ada_set_context("3.9")
cumul = cvs_sample(bv=inventoryData["bv"], id=inventoryData["item"])
cumul.stratify(strata=3, classes=10, stratMeth="cumulative")


The classification in Table 4.6 is stored in the `classSum` attribute.


In [ ]:
ada_set_context("3.9")
cumul.field("classSumm")


The stratification summary is then:


In [ ]:
ada_set_context("3.9")
cumul.field("stratSumm")


We calculate the required sample size for a desired precision of 200,000.


In [ ]:
ada_set_context("3.9")
cumul.size(desPrec=200000).n


Select sample


In [ ]:
ada_set_context("3.9")
cumul.select(seed=12345)
cumul.sample.head()


Obtain the audit values and evaluate the sample.


In [ ]:
ada_set_context("3.9")
true_values = inventoryData.set_index("item").loc[cumul.sample["item"], "av"]
cumul.evaluate(av=true_values)


A summary table of the estimates for each of the estimation methods is stored in the `Estimates` argument, which is part of the `evalResults` argument.


In [ ]:
ada_set_context("3.9")
cumul.eval_results["Estimates"]
